# AI Agent with Tools 
This notebook demonstrates how to create an AI agent that can utilize tools to perform tasks. We will use the `tavily` library to build our agent and integrate it with various tools for enhanced functionality.

- Build an AI agent that can utilize various tools to perform tasks effectively.
- Two types of tools used viz. user tools and agent tools.

In [1]:
%pip install -q openai-agents==0.2.2 python-dotenv requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.1/161.1 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.7/150.7 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 34.4 MB/s eta 0:00:00


In [18]:
import os
import requests
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown

import json  # Import the json module for handling JSON data
from typing_extensions import TypedDict, Any  # Import TypedDict for type hinting, Any for general typing
from agents import function_tool  # Import the function_tool from the agents module

from agents import SQLiteSession # Import SQLiteSession for agent memory management
from agents import Agent, Runner # Import Agent and Runner for creating and running the AI agent

# from agents import FunctionTool, function_tool, RunContextWrapper, CodeInterpreterTool # Import agent tool decorators and helpers
from agents import FunctionTool, function_tool, CodeInterpreterTool # Import agent tool decorators and helpers

In [ ]:
# load_dotenv()
# openai_api_key = os.getenv("OPENAI_API_KEY")
# tavily_api_key = os.getenv("TAVILY_API_KEY")

# print("✅ Keys loaded:")
# print(f"OpenAI API Key: {openai_api_key[:5]}***")
# print(f"Tavily API Key: {tavily_api_key[:5]}***")

# openai_client = OpenAI(api_key=openai_api_key)

In [3]:
load_dotenv()
# Get the OpenAI API key from environment variables or prompt if missing
openai_api_key = os.getenv("OPENAI_API_KEY")
if not openai_api_key:
    from getpass import getpass
    openai_api_key = getpass("OpenAI API key (will not be echoed): ")

# Ensure the agents/OpenAI client can read the key via the environment variable
if openai_api_key:
    os.environ["OPENAI_API_KEY"] = openai_api_key
    

tavily_api_key = os.getenv("TAVILY_API_KEY")
if not tavily_api_key:
    from getpass import getpass
    tavily_api_key = getpass("Tavily API key (will not be echoed): ")

# Ensure the agents/OpenAI client can read the key via the environment variable
if tavily_api_key:
    os.environ["TAVILY_API_KEY"] = tavily_api_key

# Configure the OpenAI Client using our key
openai_client = OpenAI(api_key=openai_api_key)

print("✅ Keys loaded:")
print(f"OpenAI API Key: {openai_api_key[:5]}***")
print(f"Tavily API Key: {tavily_api_key[:5]}***")

print("OpenAI client successfully configured.")

✅ Keys loaded:
OpenAI API Key: sk-pr***
Tavily API Key: tvly-***
OpenAI client successfully configured.


In [4]:
def print_markdown(text: str):
    display(Markdown(text))

## Define TAVILY Search functions & create tool

In [5]:
# Define a TypedDict for the expected parameters for the Tavily search function
# A TypedDict is like a blueprint for a dictionary in Python
# It tells Python exactly what keys the dictionary should have and what type of values go with each key.

class TavilySearchParams(TypedDict):
    query: str         # The search query string
    max_results: int   # The maximum number of results to return

In [6]:
# Let's define a function that searches the web using the Tavily API and gives back a short summary of the top results.
# Decorate the function as a FunctionTool for OpenAI Agents SDK
@function_tool
def tavily_search(params: TavilySearchParams) -> str:
    """
    Calls the Tavily API and returns a string summary of top search results.

    Args:
        params (TavilySearchParams): Dictionary with 'query' (str) and 'max_results' (int).

    Returns:
        str: A formatted string summarizing the top search results, or an error message.
    """

    # The web address (endpoint) for sending search requests to Tavily ( Tavily API endpoint)
    url = "https://api.tavily.com/search" 

    # Tell the API that we’re sending JSON data
    headers = {"Content-Type": "application/json"}  

    # What we’re sending to the API:
    # Our secret API key (so Tavily knows it's us)
    # The search text (query)
    # How many results we want (defaults to 2 if not given)
    payload = {
        "api_key": tavily_api_key,  # Use the Tavily API key from environment
        "query": params["query"],   # The search query from params
        "max_results": params.get("max_results", 2),  # Use max_results from params, default to 2 if not provided
    }

    # Send the search request to Tavily (POST means we’re sending data)
    response = requests.post(url, json = payload, headers = headers) 

    # Check if the search worked (200 = OK)
    if response.status_code == 200:  # If the request was successful
        results = response.json().get("results", [])  # Extract the 'results' list from the response JSON
        
        # Build a summary string with each result's title and content, numbered
        summary = "\n".join([f"{i+1}. {r['title']}: {r['content']}" for i, r in enumerate(results)])
        return summary if summary else "No relevant results found."  # Return summary or fallback message
    else:
        return f"Tavily API error: {response.status_code}"  # Return error message with status code if request failed


## Build & Run AI Agent with Tools - External Search tool Tavily API

In [8]:
# Let's add memory to our agent
session = SQLiteSession("live_researcher_practice")

In [10]:
live_researcher_agent = Agent(
    name = "Live Market Researcher",
    instructions = """
        CONTEXT:
        You are a world-class market research assistant with access to real-time web search via the tavily_search tool.

        INSTRUCTION:
        - Analyze the user's question and determine if recent or real-time information is needed.
        - If the question involves recent events, news, or product info, always call tavily_search.
        - Summarize search results clearly and concisely, do not copy-paste.
        - Always start your answer with: "🔍 According to a web search …"

        INPUT:
        You will receive a conversation history and the latest user question. Use the full context to inform your response.

        OUTPUT:
        Provide a clear, well-structured answer that references the search results when appropriate. If you use tavily_search, integrate the findings into your summary.
    """,
    model = "gpt-4.1-mini",
    tools = [tavily_search])

print("✅ Agent created with Tavily tool.")

✅ Agent created with Tavily tool.


In [ ]:
# First question
q1 = "What are people saying about new Danish Government?"

print_markdown(f"**User:** {q1}")

run1 = await Runner.run(
    starting_agent = live_researcher_agent,
    input = q1,
    session = session,
)

print_markdown(f"### 🤖 Agent’s Answer\n{run1.final_output}")

**User:** What are people saying about new Danish Government?

### 🤖 Agent’s Answer
🔍 According to a web search, the new Danish government, led by Prime Minister Mette Frederiksen, has sparked various reactions and discussions. The government is a left-leaning coalition formed after months of intense negotiations, involving the Social Democrats, the Social Liberals, the Green Left, and the centrist Moderates. It aims to tackle pressing issues like the cost of living crisis by implementing policies such as halving VAT on food and offering free public transport for young people. 

People note the government's firm stance against external pressures, notably from the US regarding Greenland. However, the coalition is a minority government, and analysts express concern about its stability due to the long and difficult formation process and recent scandals that have affected Frederiksen's leadership.

The new government also reflects a complex and diverse political landscape, with 12 parties winning seats in the parliament, marking one of Denmark’s most fragmented parliaments in recent history. The formation of this government ends a period of political uncertainty and sets an agenda focusing on improving everyday lives for Danes.

In summary, public and expert opinion views the government as ambitious in social policy but cautious about its long-term durability. There is notable attention on how it will navigate internal and external challenges in the coming years.

- **Note: You can monitor all tools used by the Agent through traces, available in the OpenAI platform when you log in to your account.**
- **Link: https://platform.openai.com/logs/**

In [13]:
# First question
q2 = "What is the public opinion about the new Danish Government?"

print_markdown(f"**User:** {q2}")

run2 = await Runner.run(
    starting_agent = live_researcher_agent,
    input = q2,
    session = session,
)

print_markdown(f"### 🤖 Agent’s Answer\n{run2.final_output}")

**User:** What is the public opinion about the new Danish Government?

### 🤖 Agent’s Answer
🔍 According to the web search, public opinion about the new Danish government appears mixed. While there is some optimism about the government's focus on easing the cost of living with measures like halving VAT on food and free public transport for youth, there are also concerns. The coalition's formation was historically difficult and prolonged, indicating a fragmented and divided political atmosphere, which may reflect some public skepticism about the government's ability to govern effectively.

Moreover, some analysts and citizens are wary due to recent scandals involving Prime Minister Mette Frederiksen and the fact that this is a minority coalition government, which often faces challenges in passing legislation and maintaining stability. The public seems aware that this government will have to navigate significant external pressures, including geopolitical issues related to Greenland.

In essence, while there is hope for positive social reform and economic relief, there is also apprehension about the government's longevity and effectiveness amid political complexities. This keeps the public opinion cautious but engaged.

**PRACTICE OPPORTUNITY SOLUTION:**
- **Modify the tavily_search function so that it returns only the titles of the search results, without including the content.**
- **Increase the number of results returned from 2 to 3 by updating the max_results parameter in the function.**

In [14]:
@function_tool
def tavily_search(params: TavilySearchParams) -> str:
    """
    Calls the Tavily API and returns a string of only the titles from the top search results.
    """
    url = "https://api.tavily.com/search"  # Tavily API endpoint
    headers = {"Content-Type": "application/json"}  # Request headers
    payload = {
        "api_key": tavily_api_key,
        "query": params["query"],  # Search query from params
        "max_results": params.get("max_results", 3)  # Default 3 results instead of 2
    }

    response = requests.post(url, json=payload, headers=headers)

    if response.status_code == 200:
        results = response.json().get("results", [])
        
        # Only return numbered titles
        summary = "\n".join([f"{i+1}. {r['title']}" for i, r in enumerate(results)])
        
        return summary if summary else "No relevant results found."
    else:
        return f"Tavily API error: {response.status_code}"


In [15]:
session = SQLiteSession("live_researcher_practice")

prod_memory = [] # This will hold our agent's memory of past interactions in this session
q1 = "What do reviewers say about the new Tesla Autodrive in Denmark?"

print_markdown(f"**User:** {q1}")
run1 = await Runner.run(live_researcher_agent, q1, session = session)
print_markdown(f"**Answer:** {run1.final_output}")

**User:** What do reviewers say about the new Tesla Autodrive in Denmark?

**Answer:** 🔍 According to a web search, Tesla's new Full Self-Driving (FSD) system, branded as Tesla Autodrive, has been provisionally approved in Denmark, making it the fourth European country to officially clear the software. The Danish Road Traffic Authority (Færdselsstyrelsen) reviewed the system thoroughly and agreed it positively contributes to road safety by assisting the driver while driving. This approval follows similar clearances in the Netherlands, Lithuania, and Estonia, bypassing slower EU-wide approval processes.

Reviewers in Denmark generally express strong positive impressions, noting that the Tesla FSD handles various driving scenarios effectively, including challenging situations like sketchy parking and dirt roads, with very few mistakes. However, there is emphasis that the system is still driver-assisted and requires the driver to remain attentive and in control at all times. The system monitors driver attention and can safely bring the car to a stop if needed.

Denmark's regulatory approval notably fast-tracks the availability of Tesla FSD there compared to other European countries, although the approval remains provisional and subject to change based on the broader EU regulatory process.

In summary, Danish reviewers and authorities see Tesla Autodrive as a significant step forward in driver assistance and safety, but it is not fully autonomous and requires active driver supervision.

In [16]:
q2 = "Summarize the main takeways from the previous question in 3 bullet points."
print_markdown(f"**User:** {q2}")
run2 = await Runner.run(live_researcher_agent, q2, session = session)
print_markdown(f"**Answer:** {run2.final_output}")

**User:** Summarize the main takeways from the previous question in 3 bullet points.

**Answer:** - Denmark has provisionally approved Tesla's Full Self-Driving (FSD) system, making it the fourth European country to do so after the Netherlands, Lithuania, and Estonia.

- Reviewers in Denmark praise Tesla Autodrive for effectively handling various driving conditions with minimal errors, but stress that it is a driver-assistance system requiring continuous driver attention and control.

- The approval bypasses slower EU-wide processes, fast-tracking Tesla FSD availability in Denmark, though the provisional status means it could be revoked depending on broader EU regulatory outcomes.

## Leverage OpenAI built-in tools for enhanced agent capabilities

- Instead of creating a tool, you can leverage existing tools available in OpenAI Agents SDK: https://openai.github.io/openai-agents-python/ref/tool/
    - **WebSearchTool:** Let your agent do a real‑time web search 
    - **FileSearchTool:** Search your vector stores for content retrieval 
    - **ComputerTool:** Have the agent take actions on your system like clicking or typing 
    - **CodeInterpreterTool, ImageGenerationTool, HostedMCPTool:** Extra built‑ins for code, visuals, MCP connections all ready to go

In [23]:
# Use the hosted CodeInterpreterTool to provide a safe, sandboxed Python execution environment for the agent.
# This tool allows the agent to run Python code in a secure container, which is isolated from the main system.
# The 'tool_config' specifies that we want an automatic container (per documentation).
code_interpreter = CodeInterpreterTool(
    tool_config = {"type": "code_interpreter", 
                   "container": {"type": "auto"}})

# Print a message to confirm that all tools (get_catalog, run_readonly_sql, code_interpreter) are ready for use.
print("✅ Tool ready: code_interpreter")

✅ Tool ready: code_interpreter


In [24]:
analyst_agent = Agent(name = "Analyst Agent",
                      instructions = """
CONTEXT:
You are a world-class market research assistant with access to both real-time web search (via the tavily_search tool) and Python code execution (via the code_interpreter tool).

INSTRUCTION:
- Carefully analyze the user's question to determine whether it requires:
    - Recent or real-time information (use tavily_search),
    - Data analysis, calculations, or code execution (use code_interpreter),
    - Or a combination of both tools.
- Use tavily_search for up-to-date facts, news, or product information.
- Use code_interpreter for tasks involving data analysis, calculations, or code-based reasoning.
- If the task benefits from both tools, use them together and integrate the results.
- Clearly summarize your findings and reasoning. Do not copy-paste search results; always paraphrase.
- When using search, begin your answer with: "🔍 According to a web search …"
- When using code, explain your process and results clearly.

INPUT:
You will receive a conversation history and the latest user question. Use the full context to decide which tool(s) to use and to inform your response.

OUTPUT:
Provide a clear, well-structured answer. Reference search results and/or code outputs as appropriate, and integrate them into your summary.
""",
    model = "gpt-5.4-mini",
    tools = [tavily_search, code_interpreter],
)

print("✅ Agent created with Tavily and Code Interpreter tools.")

✅ Agent created with Tavily and Code Interpreter tools.


In [25]:
session = SQLiteSession("live_researcher")

# New question: Ask the agent to get the current price of a Tesla Cyber Truck in Denmark,
# then simulate the price increase if tariffs rise from 5% to 20% in 1% increments,
# and calculate the new prices using Python, providing a summary across all scenarios.

q1 = (
    "Find the current price of a Tesla Cars in Denmark. "
    "Then, simulate how the price would change if import tariffs increased from 5% to 20% in 1% increments. "
    "For each tariff rate, calculate the new price using Python and provide a summary of the results across all scenarios."
)

print_markdown(f"**User:** {q1}")

run1 = await Runner.run(
    starting_agent = analyst_agent,
    input = q1,
    session = session,
)

print_markdown(f"### 🤖 Agent’s Answer\n{run1.final_output}")

**User:** Find the current price of a Tesla Cars in Denmark. Then, simulate how the price would change if import tariffs increased from 5% to 20% in 1% increments. For each tariff rate, calculate the new price using Python and provide a summary of the results across all scenarios.

### 🤖 Agent’s Answer
🔍 According to a web search, Tesla’s Denmark inventory and pricing vary by model and trim, but a commonly cited current ballpark price for a Tesla in Denmark is about **DKK 300,000** for a base-level model. I’ll use that as the starting price for the tariff simulation.

### Tariff simulation in Python
Using a base price of **DKK 300,000**, I calculated the new price for tariff rates from **5% to 20%** in **1% increments**.

Formula:
**New price = Base price × (1 + tariff rate)**

### Results
| Tariff | New Price (DKK) |
|---|---:|
| 5%  | 315,000 |
| 6%  | 318,000 |
| 7%  | 321,000 |
| 8%  | 324,000 |
| 9%  | 327,000 |
| 10% | 330,000 |
| 11% | 333,000 |
| 12% | 336,000 |
| 13% | 339,000 |
| 14% | 342,000 |
| 15% | 345,000 |
| 16% | 348,000 |
| 17% | 351,000 |
| 18% | 354,000 |
| 19% | 357,000 |
| 20% | 360,000 |

### Summary across scenarios
- **Lowest price:** DKK **315,000** at 5%
- **Highest price:** DKK **360,000** at 20%
- **Total increase across the range:** DKK **45,000**
- **Average price across all scenarios:** DKK **337,500**

If you want, I can also redo this using a **specific Tesla model in Denmark** (for example Model 3, Model Y, or Model S) instead of the generic base estimate.